### RAG란?
- LLM 에게 필요한 자료를 주고 답변하게 시키는 것
- LLM은 똑똑하지만 모르는 걸 지어내는 경향 있음 + 최신정보 없고, 사내, 비밀 문서는 없다
- 관련 질문을 했을 경우, 검색으로 근거를 찾아서 필요 자료를 줘야 하는 상황

#### RAG 검색의 장점
- 환각 감소
- 최신성 : 최신 문서를 검색에 넣어서 질문하게 시킬 수 있음
- 출처 : 어떤 문서를 근거로 했는지 밝힐 수 있음
---
- 챗봇, 문서 검색, 상담 시스템 등의 표준 구조!

### RAG 흐름
1. 검색 -> 벡터 저장소
2. 증강 -> 찾은 문서를 프롬프트에 넣으면 됨. 프롬프트 최적화에 시간 많이 걸림!
3. 생성

In [1]:
import pandas as pd

df = pd.read_csv("../data/11-1_뉴스정제.csv").head(50)
df.head(3)

,제목,본문,카테고리,요약,출처URL,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제,"6 6일 현대백화점그룹이 광주시에 문화복합몰을 만든다고 6일 밝혔으며, 광주시는 서...",https://n.news.naver.com/mnews/article/001/001...,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직...,경제,이이스항공은 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 출소한 것...,https://n.news.naver.com/mnews/article/003/001...,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 ‘10주년 기념주...,경제,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 소셜미디어 인스타...,https://n.news.naver.com/mnews/article/366/000...,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...


In [2]:
docs = df['정제본문'].tolist()

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [10]:
import numpy as np

# 임베딩 함수 만들기
def embed(texts):
    """텍스트 목록을 받아서 벡터 배열로 변환하는 함수. openai embedding을 사용해 문서 수 x 1536차원으로 변환"""

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    return np.array([item.embedding for item in response.data])

In [11]:
doc_vecs = embed(docs)
doc_vecs[0]

array([-0.0206604 , -0.00285912,  0.03421021, ...,  0.00055361,
       -0.02098083, -0.00829315], shape=(1536,))

### 1단계 - 질문으로 관련문서 검색하기
- 질문을 먼저 임베딩

In [12]:
question = "리벨리온이라는 회사는 어떤 회사야?"

# 테스트용으로 근거 없이 답변 받아보기

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{"role" : "user", "content" : question}]
)

print(response.choices[0].message.content)

리벨리온(Rebellions)은 **인공지능(AI) 반도체를 설계하는 한국의 팹리스 스타트업**입니다. 직접 반도체 공장을 운영하기보다는 칩을 설계하고, 생산은 TSMC 같은 파운드리에 맡기는 방식입니다.

### 주요 사업
- **AI 가속기·NPU 설계**
  - 데이터센터에서 대규모 AI 모델을 학습·추론할 때 필요한 연산을 빠르고 전력 효율적으로 처리하는 칩입니다.
- **국산 AI 반도체 개발**
  - 엔비디아 GPU 의존도를 낮추고, 한국 기업의 AI 인프라 경쟁력을 높이는 것을 목표로 합니다.
- **데이터센터·클라우드용 솔루션**
  - 생성형 AI, 대규모 언어 모델, 영상·음성 처리 등에 활용될 수 있습니다.

### 대표 제품
- **ATOM(아톰)**: 생성형 AI 추론에 초점을 둔 데이터센터용 AI 반도체
- **ION(아이온)**: 초기 AI 가속기 제품
- 이후에는 여러 칩을 묶어 대규모 AI 서비스에 대응하는 제품군도 개발하고 있습니다.

### 회사 특징
리벨리온은 2020년 설립됐으며, 엔비디아 GPU와 정면으로 모든 영역에서 경쟁하기보다는 **AI 추론 작업에서 높은 성능과 전력 효율을 제공하는 것**을 핵심 전략으로 삼고 있습니다. 특히 전력 사용량이 큰 데이터센터에서는 칩의 성능뿐 아니라 **와트당 성능과 비용**이 중요하기 때문에 이를 차별점으로 내세웁니다.

또한 2024년에는 SK텔레콤 계열의 AI 반도체 회사인 **사피온코리아와 합병**해 기술과 사업 규모를 확대했습니다.

한마디로 정리하면, 리벨리온은 **“생성형 AI 시대의 데이터센터용 국산 AI 칩을 만드는 회사”**라고 볼 수 있습니다. 다만 엔비디아처럼 GPU 생태계 전체를 갖춘 기업은 아니므로, 앞으로는 칩 성능뿐 아니라 소프트웨어·개발도구·고객 확보가 중요한 과제입니다.


In [13]:
query_vec = embed([question])[0]   # 리스트로 만드는 이유: 아까 함수를 여러개 동시에 처리받도록 해놨기 때문
query_vec                          # 그리고 한 질문에 대한 답변 가져오려면 [0]으로 가져와야 함!

array([ 0.04122925, -0.03643799,  0.02145386, ...,  0.01843262,
       -0.01194   ,  0.01235962], shape=(1536,))

In [ ]:
# 50개 전부와 질문과 유사도 계산! 이때 임베딩 벡터는 둘 다 크기 1이므로 내적만 하면 cos_sim 나옴
similarity = doc_vecs @ query_vec
similarity

array([0.09070987, 0.22482695, 0.20538552, 0.09310987, 0.1961786 ,
       0.20126751, 0.07952026, 0.06839526, 0.15578684, 0.33702296,
       0.23635968, 0.1986368 , 0.14350459, 0.23317709, 0.24632622,
       0.26861323, 0.16349492, 0.228692  , 0.24104356, 0.2822263 ,
       0.14546497, 0.14209805, 0.23273935, 0.14293291, 0.18489471,
       0.07043638, 0.17240487, 0.2178767 , 0.20545104, 0.18157495,
       0.13165265, 0.15408732, 0.17032746, 0.22611253, 0.06339709,
       0.16133664, 0.04637148, 0.18075829, 0.19603426, 0.30023079,
       0.12269022, 0.2641298 , 0.10721175, 0.22673048, 0.13451433,
       0.23621555, 0.2753476 , 0.2338715 , 0.29268852, 0.19096995])

In [ ]:
# 이제 유사도 가장 높은 것 찾기

top = pd.Series(similarity).sort_values(ascending=False).head(5)    # 가장 유사도 높은 문서 df로 만들어 상위 5개
top

9     0.337023
39    0.300231
48    0.292689
19    0.282226
46    0.275348
dtype: float64

In [ ]:
for i, score in top.items():
    print (i, score, docs[i][:50])  # 가장 유사한 문서 인덱스 받아서 앞의 50자만

9 0.33702295729023035 국내 AI반도체 스타트업 리벨리온에 300억원 투자 AI반도체 영역 본격 진입 외산 GPU
39 0.30023079071602865 LG유플러스 LG전자 LG생활건강은 LG그룹 창립 75주년을 기념해 공동 이벤트 함께 걸어
48 0.2926885166059492 LG전자와 SM엔터테인먼트가 홈트레이닝 시장 공략에 나선다 사진은 조주완 LG전자 사장 사
19 0.2822262951011112 신한금융투자 보고서 이데일리 이은정 기자 증시 급락세가 이어진 가운데 2분기 실적시즌이 다
46 0.27534760422798854 슈나이더 일렉트릭이 스마트팩토리 솔루션으로 생산성 향상은 물론 탄소 중립에 앞장선다 탄소 


### 2, 3단계
- 문서를 프롬프트에 넣기

In [ ]:
context = ""

for i in top.index:
    context += docs[i] + "\n\n"

print(context) 
# 유사도 높더라도 전부 리벨리온 관련있는 건 아니다! 관련은 1개만

국내 AI반도체 스타트업 리벨리온에 300억원 투자 AI반도체 영역 본격 진입 외산 GPU 의존도 극복 AI반도체 사업 진출 넘어 국내 생태계 조성 국내서 AI 풀스택 확보 초대규모 GPU팜 조성 후 전용 AI반도체 국산화 글로벌 진출 기반 마련 리벨리온 KT 최적의 파트너 KT 리벨리온 AI 반도체 사업 로드맵 KT 제공 파이낸셜뉴스 KT가 리벨리온과 손잡고 국산 AI 반도체를 이용해 초대규모 GPU팜을 구축하는 등 국가 AI 경쟁력을 강화에 나섰다 KT는 6일 리벨리온에 300억원 규모의 전략적 투자를 단행하고 사업 협력에 나선다고 밝혔다 인공지능 AI 반도체 시장은 2030년 1179달러 약 152조1660억원 에 달할 것으로 전망되고 있다 AI원팀으로 외산 의존도 KT는 이번 리벨리온과 협력으로 외국산 AI 반도체 의존도를 줄여 나갈 계획이다 리벨리온은 지난달에도 620억원 규모의 시리즈 A 투자를 유치한 주문형 반도체 ASIC 설계에 특화된 국내 AI 반도체 설계 팹리스 스타트업이다 현재 AI 서비스 개발에 필요한 컴퓨팅 인프라 영역에서 엔비디아 등 외국산 GPU 그래픽 처리 장치 점유율이 80 를 육박하고 있다 이는 지금까지는 대부분의 AI서비스 솔루션이 엔비디아가 제공하는 SW CUDA를 기반으로 개발돼 대부분 AI 반도체 개발사들도 엔비디아 의존도를 떨치기 어려웠다 이에 KT는 AI원팀으로써 협력 중인 스타트업들과 함께 국내 AI풀스택을 구축키로 했다 우선 연내 수천장 규모에 달하는 초대규모 GPU팜을 구축한다 내년에는 해당 GPU팜에 하이퍼스케일 AI컴퓨팅 HAC 전용으로 자체 개발한 AI 반도체를 접목할 예정이다 이 AI 반도체는 AI알고리즘에 최적화된 신경망처리장치 NPU 다 NPU는 GPU 대비 3배 넘는 에너지 효율과 저렴한 도입 비용 복잡한 알고리즘에도 적합한 성능 등이 강점이다 이미 KT는 지난해 kt 클라우드가 출시한 종량제 GPU 서비스 HAC에 CUDA를 지원할 수 있는 자체 AI 프레임워크 적용에 성공했다 이를 기반으로 엔비디

In [22]:
prompt = f"""
아래 [근거 자료]만 참고해서 질문에 답하세요.
자료에 없으면 '자료에 없음' 이라고 답하세요

[근거 자료]
{context}

[질문]
{question}
"""

response_rag = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[{"role" : "user", "content" : prompt}]    # 질문 prompt 전부 넣음!
)

print(response_rag.choices[0].message.content)

리벨리온은 **주문형 반도체(ASIC) 설계에 특화된 국내 AI 반도체 팹리스 스타트업**입니다. AI 알고리즘에 최적화된 **신경망처리장치(NPU)** 등을 개발하며, 외산 GPU 의존도를 줄이고 국산 AI 반도체 생태계를 구축하는 것을 목표로 합니다.
